In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import warnings
from collections import Counter
warnings.filterwarnings('ignore')

In [ ]:
# Configuration
pd.set_option('display.max_columns', None)

In [ ]:
# Load data
current_user = os.getlogin()
DATA_DIR = Path(f"/home/{current_user}/local-share")
OUT_DIR = DATA_DIR / "processed"
combined_path = OUT_DIR / "v2_combined.csv"

data = pd.read_csv(combined_path, sep=None, engine="python", encoding="utf-8-sig")
print(f"Loaded data shape: {data.shape}")
data.head()

## SDO78 (100 Dagen Monitor) Integration

The sdo78 dataset contains responses to the freshmen questionnaire filled out after ~100 days of studying. It is merged using a **left join** on `SINH_ID` so that all students are retained regardless of whether they responded. Non-responders receive `NaN` for all question columns, which is handled in notebook 04 (missing value imputation) using cohort-wise median imputation; the `sdo78_response_type` column encodes response completeness (Complete-responder / Partial-responder / Non-responder).

In [ ]:
# Load sdo78 data
sdo78_path = OUT_DIR /  "sdo78_combined.csv"
sdo78_df = pd.read_csv(sdo78_path, sep=None, engine="python", encoding="utf-8-sig")

# Left join sdo78 on data using SINH_ID
data = data.merge(sdo78_df, left_on="SINH_ID", right_on="sdo78_SINH_ID", how="left")

# 1. Initial Data Exclusions
Exclude college years 2018-2020 due to missing orientation data (storage started in 2020).

In [ ]:
# Exclude college years 2018-2020
data = data[~data['sdo5_degree_COLLEGEJAAR'].isin([2018, 2019, 2020])].reset_index(drop=True)
print(f"Size of dataset after year exclusion: {data.shape}")

## 2. Remove Unnecessary Columns

In [ ]:
# Drop ID, code and other redundant columns (except SINH_ID primary key)
cols_to_remove = [
    "sdo1_prev_distance_id",
    "sdo1_postal_distance_id",
    "sdo1_prev_distance_postcode",
    "sdo1_postal_distance_postcode",
    "sdo1_prev_distance_distance_to_3812_PA",
    "sdo1_postal_distance_distance_to_3812_PA",
    "sdo2_orientation_STUDENTNUMMER",
    "sdo5_degree_degree_code_letters",
    "sdo5_degree_degree_code",
    "sdo5_degree_POSTCODE",
    "sdo1_previous_Previous_School_Postal_Code",
    "sdo6_results_Total_Credits_Block_B",         # Exclude block B results
    "sdo6_results_Potential_Credits_B",           # Exclude block B results
    "sdo6_results_Percentage_Credits_B",          # Exclude block B results
    "sdo6_results_Average_Grade_B",               # Exclude block B results
    "sdo5_degree_drop_out_temporary",             # Exclude detailed drop-out type
    "sdo5_degree_drop_out_to_other_degree_in_HU", # Exclude detailed drop-out type
    "sdo5_degree_drop_out_without_propedeuse",    # Exclude detailed drop-out type
    "sdo5_degree_drop_out_with_propedeuse",        # Exclude detailed drop-out type
    "sdo78_SINH_ID",
    "sdo78_year"
]

data = data.drop(columns=cols_to_remove, errors='ignore')
print(f"After dropping {len(cols_to_remove)} columns: {data.shape}")

In [ ]:
# Check for duplicate columns
identical_columns = [
    (col1, col2) for i, col1 in enumerate(data.columns) 
    for col2 in data.columns[i+1:] 
    if data[col1].equals(data[col2])
]

if identical_columns:
    print("Identical columns found:", identical_columns)
else:
    print("✓ No identical columns")

In [ ]:
# Remove single-value columns
single_value_cols = [col for col in data.columns if data[col].nunique() == 1]
if single_value_cols:
    print(f"Dropping {len(single_value_cols)} single-value columns:", single_value_cols)
    data = data.drop(columns=single_value_cols)
else:
    print("✓ No single-value columns")

## 3. Date Processing
### 3.1 Fix corrupted dates

In [ ]:
# Fix corrupted dates with hidden YYYYMMDD format after '0.'
def repair_hidden_ymd(raw: pd.Series) -> pd.Series:
    """Extract YYYYMMDD hidden after '0.' (e.g., 1970-...0.020180901) and parse."""
    s = raw.astype(str)
    frac = s.str.extract(r'0\.(\d{8,9})')[0]
    ymd = frac.str.lstrip('0').str[-8:]
    
    repaired = pd.to_datetime(ymd, format='%Y%m%d', errors='coerce')
    normal = pd.to_datetime(s, errors='coerce')
    out = repaired.combine_first(normal)
    
    # Filter valid years and remove Unix epoch artifacts
    year_ok = out.dt.year.between(1990, 2035, inclusive="both")
    out = out.where(year_ok, pd.NaT)
    out = out.mask(out.dt.date.eq(pd.Timestamp("1970-01-01").date()), pd.NaT)
    return out

# Apply to corrupted date columns
buggy_cols = ["sdo5_degree_enrollment_date", "sdo5_degree_degree_start_date", 
              "sdo5_degree_degree_end_date", "dropout_date"]
for col in buggy_cols:
    if col in data.columns:
        data[col] = repair_hidden_ymd(data[col])

# Parse Final Exam Date (YYYY-MM-DD format)
data['sdo1_previous_Final_Exam_Date'] = pd.to_datetime(data['sdo1_previous_Final_Exam_Date'], 
                                                        format='%Y-%m-%d', errors='coerce')

# Parse other date columns normally
other_date_cols = [ "sdo2_skc_SKC_DATUM", "sdo2_orientation_First_Event_Date", "sdo2_orientation_Last_Event_Date", "sdo5_degree_de_enrollment_date",]
for col in other_date_cols:
    if col in data.columns:
        data[col] = pd.to_datetime(data[col], errors="coerce", dayfirst=True)

print("Date columns processed successfully")


### 3.2 Create Unified Dropout Date
Combine two dropout date sources:
- `sdo5_degree_de_enrollment_date`: drops out during the year
- `sdo5_degree_degree_end_date`: doesn't re-register for next year

In [ ]:
# Create unified dropout date
data['dropout_date'] = data['sdo5_degree_de_enrollment_date'].fillna(data['sdo5_degree_degree_end_date'])

# Drop original date columns
data = data.drop(columns=['sdo5_degree_de_enrollment_date', 'sdo5_degree_degree_end_date'], errors='ignore')
print(f"Final data shape after cleaning: {data.shape}")

# Look at dropout date for all drop-outs
dropout_dates = data.loc[data['sdo5_degree_drop_out'] == 1, 'dropout_date']
dropout_dates.head(10)

### 3.3 Clean SKC date
SKC advice can be giving after the degree start date, but it is often an administrative solution. 


In [ ]:
# When SKC advice date is after the degree start, we replace the SKC date with degree start date.
skc_col = 'sdo2_skc_SKC_DATUM'
start_col = 'sdo5_degree_degree_start_date'
if skc_col in data.columns and start_col in data.columns:
    skc_after_start = data[skc_col] > data[start_col]
    data.loc[skc_after_start, skc_col] = data.loc[skc_after_start, start_col]
    print(f"Replaced {skc_after_start.sum()} SKC dates that were after degree start date.")

### 3.4 Clean number of orientation event types
Data quality issue: there exist a large number of rows where number of orientation events > 0, but where number of orientation event types = 0. We set these 0 to NA. 

In [ ]:
# For sdo2_orientation_Number_of_Event_Types values 0, replace with NaN
data['sdo2_orientation_Number_of_Event_Types'] = data['sdo2_orientation_Number_of_Event_Types'].replace(0, np.nan)

## 4. Clean Categorical Columns

### Degree Programs

In [ ]:
# Clean degree program names
def clean_degree(value):
    if pd.isna(value):
        return "Unknown"
    
    v = str(value).strip()
    
    # Exact mappings for specific cases
    exact_map = {
        "B Education in Primary Schools (age 4 - 12) (day)": "Primary_Education_Day",
        "B Education in Primary Schools (age 4 - 12) (ALPO)": "Primary_Education_ALPO",
        "B Education in Primary Schools (age 4 - 12) (Regular)": "Primary_Education_Regular",
        "B Arts Therapies (Music Therapy)": "Music_Therapy",
        "B Arts Therapies (Drama Therapy)": "Drama_Therapy",
        "B Arts Therapies (Art Therapy)": "Art_Therapy",
    }
    if v in exact_map:
        return exact_map[v]
    
    # Remove prefixes and standardize
    v = re.sub(r"^\s*(B|Bachelor of)\s+", "", v, flags=re.IGNORECASE)
    v = v.replace("Chemics", "Chemistry").replace("&", " ")
    v = re.sub(r"\b(?:and|with|in)\b", "", v, flags=re.IGNORECASE)
    v = v.replace(",", " ")
    v = re.sub(r"\s+", " ", v).strip().title().replace(" ", "_")
    v = re.sub(r"_+", "_", v).strip("_")
    
    return v

data['sdo5_degree_degree'] = data['sdo5_degree_degree'].apply(clean_degree)
print(f"Cleaned degree programs: {data['sdo5_degree_degree'].nunique()} unique values")

### Gender

In [ ]:
# Clean gender: exclude 'O'/'o', fill NaN with 'Unknown'
col = "sdo1_characteristics_gender"
data = data[~data[col].isin(['O', 'o'])]
data[col] = data[col].fillna("Unknown")
print(f"Gender distribution:\n{data[col].value_counts()}")

### Previous Education Type

In [ ]:
# Clean previous education type
col = "sdo1_previous_Previous_Education_Type"
data[col] = data[col].replace({"$": "Unknown"}).fillna("Unknown")
print(f"Previous education types: {data[col].nunique()} unique values")

### SKC Advice
SKC advice 'Niet_deelgenomen' almost never occurs and functionally means the same as missing data, so we set missing values also to 'Niet_deelgenomen'. 

In [ ]:
# Clean SKC advice: replace spaces with underscores, fill NaN
col = "sdo2_skc_ADVIES_DEF"
data[col] = data[col].apply(lambda x: "_".join(str(x).split()) if pd.notna(x) else "Niet_deelgenomen")
print(f"SKC advice categories: {data[col].nunique()} unique values")

### Orientation Event Types

In [ ]:
# =============================================================================
# Cleaning_sdo2_orientation_Event_Types_Attended missing valuoes
# -----------------------------------------------------------------------------
# PURPOSE
# Clean the column: sdo2_orientation_Event_Types_Attended
#
# Steps:
# 1) Fill missing with 'absent', lowercase, and strip spaces.
# 2) Print category counts.
# =============================================================================

# Fill missing with 'absent', lowercase, and strip spaces.
data["sdo2_orientation_Event_Types_Attended"] = data["sdo2_orientation_Event_Types_Attended"].fillna("orientation_type_absent").astype(str).str.strip().str.lower() 

In [ ]:
# =============================================================================
# Translate English event type names to Dutch in sdo2_orientation_Event_Types_Attended
# -----------------------------------------------------------------------------
# PURPOSE
# Standardize orientation event types by translating English terms to Dutch:
#   - "open day" → "open dag"
#   - "guided tour" → "rondleiding"
#   - "orientation day" → "proefstuderen"
#
# This ensures consistency across all event type values for analysis.
# =============================================================================

translation_map = {
    "open day": "open dag",
    "guided tour": "rondleiding",
    "orientation day": "proefstuderen"
}

def translate_event_types(event_str):
    if pd.isna(event_str):
        return event_str
    event_list = [etype.strip().lower() for etype in event_str.split(',')]
    translated_list = [translation_map.get(etype, etype) for etype in event_list]
    return ', '.join(translated_list)

data['sdo2_orientation_Event_Types_Attended'] = data['sdo2_orientation_Event_Types_Attended'].apply(translate_event_types)

# Verify translation by counting event types
event_types = data['sdo2_orientation_Event_Types_Attended'].dropna().str.split(',')
all_event_types = [etype.strip().lower() for sublist in event_types for etype in sublist]
event_type_counts = Counter(all_event_types)
print("Event type counts after translation:")
event_type_counts

## 5. Final Checks and Export

In [ ]:
# Summary statistics
print(f"Final data shape: {data.shape}")
print(f"\nMissing values (top 10):")
print(data.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# Save cleaned data
output_path = OUT_DIR / "cleaned_original.csv"
data.to_csv(output_path, index=False)
print(f"✅ Cleaned data saved to: {output_path}")